# MMA3001 Project: training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plap0404/MMA3001-Project/blob/main/notebooks/train_colab.ipynb)

Code is developed and tested locally, then run here because training needs a GPU.
This notebook rebuilds the whole pipeline from the GitHub repository, so every run
starts from the same committed code:

1. check a GPU is attached;
2. clone the repository;
3. install dependencies;
4. download the dataset (Roboflow Version 1) and filter it to 4 classes;
5. verify the dataset and run the tests;
6. record software versions;
7. run a 1-epoch smoke test, saving results to Google Drive.

**Before running (once per Colab account):**

* *Runtime -> Change runtime type -> T4 GPU.*
* Add the Roboflow API key as a Colab secret: click the **key icon** in the left
  sidebar, *Add new secret*, name `ROBOFLOW_API_KEY`, paste the key, and switch
  **Notebook access** on. The key is never written into this notebook.

Colab deletes everything in `/content` when the session ends, so anything worth
keeping is saved to Google Drive.

## 1. Check the GPU

In [ ]:
# Should list a Tesla T4 (or similar). If it errors, the runtime has no GPU.
!nvidia-smi

## 2. Clone the repository
Re-running this cell pulls the latest commits instead of cloning again.

In [ ]:
import os

REPO_URL = "https://github.com/plap0404/MMA3001-Project.git"
REPO_DIR = "/content/MMA3001-Project"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --oneline   # the exact commit this run used; record it with any results

## 3. Install dependencies
`ultralytics` provides the YOLO models. PyTorch is already installed on Colab.

In [ ]:
!pip install -q -r requirements.txt ultralytics

## 4. Download and filter the dataset

In [ ]:
from google.colab import userdata

# download_data.py reads ROBOFLOW_API_KEY from the environment
# (python-dotenv does not override a variable that is already set).
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

!python scripts/download_data.py
!python scripts/filter_labels.py

## 5. Verify the dataset and run the tests

Expected (matches the laptop, 18 Sep 2026):

| Split | Images | Empty labels | loose-meat | packaging-error | twisted-meat | unsealed |
|---|---|---|---|---|---|---|
| train | 1,752 | 20 | 127 | 1,372 | 236 | 479 |
| valid | 115 | 0 | 5 | 45 | 5 | 88 |
| test | 76 | 1 | 3 | 29 | 7 | 52 |

If anything differs, stop: the data is not what was verified locally.

In [ ]:
!python scripts/dataset_stats.py data/pork_v4class
!python -m pytest -q

## 6. Record software versions
Copy this output into PROJECT_LOG.md; it is part of the reproducibility record.

In [ ]:
import platform
import torch
import ultralytics

print("Python     ", platform.python_version())
print("PyTorch    ", torch.__version__)
print("Ultralytics", ultralytics.__version__)
print("GPU        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 7. Smoke test (1 epoch)

This is **not** the baseline. It only proves the pipeline runs end to end:
the dataset loads, training starts on the GPU, and results are saved to Drive.
Its scores are meaningless after one epoch.

In [ ]:
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/MMA3001/runs"

# Absolute path on purpose: with a relative path, Ultralytics looks for the
# dataset inside its own datasets folder instead of this repository.
DATA_YAML = str(Path("data/pork_v4class/data.yaml").resolve())

model = YOLO("yolov8n.pt")  # small pretrained model; downloads automatically
model.train(
    data=DATA_YAML,
    epochs=1,
    imgsz=640,
    batch=16,
    seed=0,
    project=RUNS_DIR,
    name="smoke_test",
    exist_ok=True,
)